# BSV Batch Processor — Table/Text GT (Docling)

Runs table extraction using Docling (not pdfplumber) across the 2,404-PDF
reserved GT pool, specifically for text/table pairs. Image extraction is
skipped entirely, this run is table-focused only.

Output:
- `bsv_dataset_gt_tables_docling.jsonl`: one JSON record per bulletin section, one line per record
- `bsv_pairs_gt_texttable_900.jsonl`: sampled text_table pairs for manual validation

**Prerequisites:** run `bsv_parser_subsections_v3.ipynb` first (or ensure
`bsv_parser.py`, including the `docling_extract_tables` function, is in the
same directory). Requires `docling` and `easyocr` installed
(`pip install docling easyocr --break-system-packages`).


In [ ]:
# Resolve paths from the repository root so notebooks work regardless of the kernel CWD.
from pathlib import Path
import os

def _find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "manifests" / "corpus_manifest.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the PhytoPairs repository root (expected data/manifests/corpus_manifest.csv)."
    )

PROJECT_ROOT = _find_project_root()
BSV_ROOT = PROJECT_ROOT / "3.Dataset Construction"
FINETUNING_ROOT = PROJECT_ROOT / "4.Finetuning"
CASE_STUDY_ROOT = PROJECT_ROOT / "5.BBCH case study"
RESULTS_RUNS_ROOT = PROJECT_ROOT / "results" / "runs" / "section_4"
DEFAULT_HF_CACHE = os.environ.get(
    "HF_HOME", str(Path.home() / ".cache" / "huggingface")
)


## 1. Install dependencies

In [1]:
import sys
!{sys.executable} -m pip install pymupdf pdfplumber --break-system-packages -q
print("Done")


Done


Shuffling the ~20,000 PDF URLs

In [2]:
"""
import random

url_file = str(BSV_ROOT / 'PDFs Download' / 'PDF_URLS.txt')

with open(url_file) as f:
    urls = f.readlines()

random.seed(42)
random.shuffle(urls)

with open(url_file, "w") as f:
    f.writelines(urls)

print(f"Shuffled {len(urls)} URLs")
"""

'\nimport random\n\nurl_file = "/projects/agrifutur/yabusaqe/BSV Sample Construction/PDF_URLS.txt"\n\nwith open(url_file) as f:\n    urls = f.readlines()\n\nrandom.seed(42)\nrandom.shuffle(urls)\n\nwith open(url_file, "w") as f:\n    f.writelines(urls)\n\nprint(f"Shuffled {len(urls)} URLs")\n'

## 2. Imports

In [3]:
import os
print(os.getcwd())
print(os.listdir(os.getcwd()))

/projects/agrifutur/yabusaqe/BSV Sample Construction
['test_bulletin.pdf', '.ipynb_checkpoints', 'bsv_parser.ipynb', '__pycache__', 'PDF_URLS.txt', 'Previous Iterations Data', 'bsv_heuristics_analysis.ipynb', 'bsv_download.ipynb', 'bsv_parser_subsections.ipynb', 'bsv_parser_subsections_v2.ipynb', 'french_words.txt', 'recall_approx.py', 'reserve_gt_pdfs.py', 'gt_reserved_pdfs.jsonl', 'Old Viewers', 'bsv_batch.ipynb', 'bsv_pair_constructor.py', 'bsv_parser.py', 'bsv_parser_subsections_v3.ipynb', 'generate_reviewer.py', 'bsv_images_gt_2404', 'bsv_batch_tables_docling.ipynb', 'bsv_images_gt_tables_docling', 'bsv_dataset_gt_tables_docling.jsonl', 'bsv_pairs_gt_texttable_900.jsonl', 'generate_reviewer_V2.py', 'bsv_dataset_gt_tables_realspans.jsonl', 'processed_urls_gt_tables_docling.txt', 'bsv_pairs_gt_texttable_partial.jsonl', 'bsv_reviewer_texttable_partial.html', 'bsv_validations_export.json', 'bsv_dataset_gt_2404.jsonl', 'bsv_pairs_gt_textimage_900.jsonl', 'processed_urls_gt_2404.txt']


In [4]:
import json, os

nb = json.load(open(str(BSV_ROOT / 'Parsers' / 'bsv_parser_subsections_v3.ipynb')))
code_cells = [c for c in nb["cells"] if c["cell_type"] == "code"]

with open(str(BSV_ROOT / "Parsers" / "bsv_parser.py"), "w") as f:
    for cell in code_cells:
        src = "".join(cell["source"])
        # skip install and test cells
        if any(skip in src for skip in [
            "pip install",
            "parse_pdf(TEST_PDF",
            "urllib.request.urlretrieve",
            "fitz.open(TEST_PDF",
            "fitz.open(TEST_CHARENTES",
            "parse_structure(TEST_PDF",
            "os.listdir(PDF_DIR)",
            "parse_pdf(TEST_PDF2",
            "parse_pdf(TEST_CHARENTES",
            "print(SIZE_THRESHOLD)",
]):
            continue
        f.write(src + "\n\n")

print("Created bsv_parser.py")
print((BSV_ROOT / "Parsers" / "bsv_parser.py").is_file())

Created bsv_parser.py
True


In [5]:
import sys
if "bsv_parser" in sys.modules:
    import importlib
    import bsv_parser
    importlib.reload(bsv_parser)
print("bsv_parser reloaded" if "bsv_parser" in sys.modules else "bsv_parser not yet imported")

bsv_parser not yet imported


In [6]:
"""
import json, os

nb = json.load(open("bsv_parser.ipynb"))
code_cells = [c for c in nb["cells"] if c["cell_type"] == "code"]

with open(str(BSV_ROOT / "Parsers" / "bsv_parser.py"), "w") as f:
    for cell in code_cells:
        src = "".join(cell["source"])
        if any(skip in src for skip in [
            "pip install",
            "TEST_PDF",
            "Bold size distribution",
        ]):
            continue
        f.write(src + "\n\n")

print("Done")
fns = __import__("re").findall(r'^def (\w+)', open("bsv_parser.py").read(), __import__("re").MULTILINE)
print(fns)
"""

<>:1: SyntaxWarning: invalid escape sequence '\w'
<>:1: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_11176/2007335901.py:1: SyntaxWarning: invalid escape sequence '\w'
  """


'\nimport json, os\n\nnb = json.load(open("bsv_parser.ipynb"))\ncode_cells = [c for c in nb["cells"] if c["cell_type"] == "code"]\n\nwith open("bsv_parser.py", "w") as f:\n    for cell in code_cells:\n        src = "".join(cell["source"])\n        if any(skip in src for skip in [\n            "pip install",\n            "TEST_PDF",\n            "Bold size distribution",\n        ]):\n            continue\n        f.write(src + "\n\n")\n\nprint("Done")\nfns = __import__("re").findall(r\'^def (\\w+)\', open("bsv_parser.py").read(), __import__("re").MULTILINE)\nprint(fns)\n'

In [7]:
import re
with open("bsv_parser.py") as f:
    src = f.read()
fns = re.findall(r'^def (\w+)', src, re.MULTILINE)
print(fns)

['is_footer', 'is_bad_header', 'is_credit_line', 'is_chart_noise', 'extract_metadata', 'parse_structure', 'extract_text', 'split_into_sections', 'extract_images', 'match_captions', 'count_raw_images', 'count_raw_tables', 'repair_table_via_docx', 'extract_tables', '_get_docling_converter', 'docling_extract_tables', 'assign_to_sections', 'parse_pdf']


This cell overrides the parser and loosens it 

In [8]:
import os, re, sys, json, time, threading, tempfile, urllib.request
import pdfplumber
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
sys.path.insert(0, str(BSV_ROOT / "Parsers"))
import bsv_parser
from bsv_parser import parse_pdf
bsv_parser.MIN_IMAGE_AREA = 30000   # was 200,000, loosened for GT only
import warnings
warnings.filterwarnings("ignore", message=".*pin_memory.*")
_strict_parse_structure = bsv_parser.parse_structure   # save the real one before overwriting

def loose_parse_structure(pdf_path, table_bboxes):
    s_raw, i_raw, text_els = _strict_parse_structure(pdf_path, table_bboxes)

    import fitz
    doc = fitz.open(pdf_path)
    total_pages = doc.page_count

    if total_pages >= 2:
        last_page_num = total_pages
        page = doc[last_page_num - 1]
        page_height = page.rect.height

        already_seen_xrefs = {img["xref"] for img in i_raw}

        img_info_list = {item["xref"]: item for item in page.get_image_info(hashes=False, xrefs=True)}
        seen_here = set()
        for img in page.get_images(full=True):
            xref = img[0]
            if xref in already_seen_xrefs or xref in seen_here:
                continue
            seen_here.add(xref)
            info = img_info_list.get(xref)
            if not info or not info.get("bbox"):
                continue
            bbox = info["bbox"]
            top, bottom = bbox[1], bbox[3]

            if top >= 0.90 * page_height:
                continue

            i_raw.append({
                "page": last_page_num, "top": top, "bottom": bottom,
                "left": bbox[0], "right": bbox[2],
                "xref": xref, "page_height": page_height,
            })

    doc.close()

    # Drop any image sitting in the bottom ~10% of its OWN page, on every
    # page, not just the last one. Catches recurring footer banners that
    # don't share one xref across pages, so the strict parser's xref-based
    # recurring-logo check never saw them in the first place.
    i_raw = [
        img for img in i_raw
        if not (img.get("page_height") and img["top"] >= 0.90 * img["page_height"])
    ]

    return s_raw, i_raw, text_els


bsv_parser.parse_structure = loose_parse_structure

# TODAY: use Docling for table extraction instead of pdfplumber, since
# it fixes the fragmentation issue pdfplumber+repair couldn't solve.
# This run is specifically for text/table pairs, not images.
bsv_parser.extract_tables = bsv_parser.docling_extract_tables

def skip_images(pdf_path, out_dir, bulletin_id, i_raw, text_els):
    return []

bsv_parser.extract_images = skip_images

import os
os.environ['SKIP_OCR'] = '1'

/projects/agrifutur/yabusaqe/BSV Sample Construction
['test_bulletin.pdf', '.ipynb_checkpoints', 'bsv_parser.ipynb', '__pycache__', 'PDF_URLS.txt', 'Previous Iterations Data', 'bsv_heuristics_analysis.ipynb', 'bsv_download.ipynb', 'bsv_parser_subsections.ipynb', 'bsv_parser_subsections_v2.ipynb', 'french_words.txt', 'recall_approx.py', 'reserve_gt_pdfs.py', 'gt_reserved_pdfs.jsonl', 'Old Viewers', 'bsv_batch.ipynb', 'bsv_pair_constructor.py', 'bsv_parser.py', 'bsv_parser_subsections_v3.ipynb', 'generate_reviewer.py', 'bsv_images_gt_2404', 'bsv_batch_tables_docling.ipynb', 'bsv_images_gt_tables_docling', 'bsv_dataset_gt_tables_docling.jsonl', 'bsv_pairs_gt_texttable_900.jsonl', 'generate_reviewer_V2.py', 'bsv_dataset_gt_tables_realspans.jsonl', 'processed_urls_gt_tables_docling.txt', 'bsv_pairs_gt_texttable_partial.jsonl', 'bsv_reviewer_texttable_partial.html', 'bsv_validations_export.json', 'bsv_dataset_gt_2404.jsonl', 'bsv_pairs_gt_textimage_900.jsonl', 'processed_urls_gt_2404.txt']
p

## 3. Configuration

In [9]:
MANIFEST     = str(BSV_ROOT / 'Ground Truth' / 'gt_reserved_pdfs.jsonl')
OUT_JSONL    = str(BSV_ROOT / 'Ground Truth' / 'bsv_dataset_gt_tables_realspans.jsonl')
OUT_IMGS     = str(BSV_ROOT / 'Ground Truth' / 'bsv_images_gt_tables_docling')
WORKERS      = 8
LIMIT        = None
CHECKPOINT   = str(BSV_ROOT / 'Ground Truth' / 'processed_urls_gt_tables_docling.txt')
MIN_PDF_SIZE = 5000

## 4. Load URLs from SPARQL results file

In [10]:
"""
import random
import time


all_pdf_files = sorted([
    f for f in os.listdir(PDF_DIR)
    if f.endswith('.pdf')
])

random.seed(time.time())
pdf_files = random.sample(all_pdf_files, min(200, len(all_pdf_files)))

print(f"Total PDFs in folder : {len(all_pdf_files)}")
print(f"Randomly sampled     : {len(pdf_files)}")
print(f"First                : {pdf_files[0]}")
print(f"Last                 : {pdf_files[-1]}")
"""

import json as _json

pdf_paths = {}
with open(MANIFEST, encoding="utf-8") as f:
    for line in f:
        entry = _json.loads(line)
        pdf_paths[entry["filename"]] = str(PROJECT_ROOT / "data" / "local" / "pdfs" / entry["filename"])

pdf_files = sorted(pdf_paths.keys())
print(f"Loaded {len(pdf_files)} reserved GT PDFs from manifest")
previous_table_dataset = BSV_ROOT / "Ground Truth" / "bsv_dataset_gt_tables_docling.jsonl"
if previous_table_dataset.is_file():
    bulletin_ids_with_tables = set()
    with previous_table_dataset.open(encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            if rec.get("tables"):
                bulletin_ids_with_tables.add(rec["bulletin_id"])
    pdf_files = [f for f in pdf_files if f.replace(".pdf", "") in bulletin_ids_with_tables]
    print(f"Scoped to {len(pdf_files)} PDFs that previously produced at least one table")
else:
    print("No previous table-extraction cache found; processing all reserved PDFs.")
if pdf_files:
    print(f"First: {pdf_files[0]}")
    print(f"Last:  {pdf_files[-1]}")

Loaded 2404 reserved GT PDFs from manifest
Scoped to 914 PDFs that previously produced at least one table
First: 07bsv_jevi_20190529_cle8bf139.pdf
Last:  spip.phpactionacceder_documentarg878clebf74b13b51cb45eee0cd755d3dd4462bdf7a6b69filepdf2Fcc_AAP_PNA-VF.pdf


## 5. Download helper

Defines `download_pdf` (retries up to `MAX_RETRIES`, checks minimum file size) and `process_one` (downloads to a temp directory, calls `parse_pdf`, returns records or empty list on failure).


In [11]:
def download_pdf(url, dest):
    for attempt in range(MAX_RETRIES + 1):
        try:
            req = urllib.request.Request(
                url, headers={"User-Agent": "BSV-Dataset-Builder/1.0"})
            with urllib.request.urlopen(req, timeout=DOWNLOAD_TIMEOUT) as resp:
                data = resp.read()
            if len(data) < MIN_PDF_SIZE:
                return False
            with open(dest, "wb") as f:
                f.write(data)
            return True
        except Exception:
            if attempt < MAX_RETRIES:
                time.sleep(2 ** attempt)
    return False


def process_one(filename):
    pdf_path = pdf_paths[filename]
    if os.path.getsize(pdf_path) < MIN_PDF_SIZE:
        return filename, []
    try:
        records = parse_pdf(pdf_path, OUT_IMGS, url=None)
        return filename, records
    except Exception as e:
        return filename, []

In [12]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU visible")

True
Quadro RTX 8000


In [13]:
import easyocr
reader = easyocr.Reader(['fr'], gpu=True)
print("EasyOCR GPU:", reader.device)

EasyOCR GPU: cuda


## 6. Run batch

 Batch run: loads the checkpoint to skip already-processed URLs. Opens `OUT_JSONL` in append mode. Runs `process_one` across remaining URLs with `ThreadPoolExecutor`. After each success, writes section records to the JSONL and appends the URL to the checkpoint under a separate lock. Prints progress every 100 PDFs and at the first 5.


In [ ]:
os.makedirs(OUT_IMGS, exist_ok=True)
write_lock      = threading.Lock()
checkpoint_lock = threading.Lock()
counts = {"processed": 0, "sections": 0, "errors": 0}

if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f:
        done_files = set(f.read().splitlines())
    print(f"Resuming: {len(done_files)} PDFs already processed")
else:
    done_files = set()

files_to_run = [f for f in pdf_files if f not in done_files]
print(f"Remaining: {len(files_to_run)} PDFs to process")

with open(OUT_JSONL, "a", encoding="utf-8") as out_f:
    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        futures = {executor.submit(process_one, fname): fname for fname in files_to_run}
        for future in as_completed(futures):
            fname, records = future.result()
            with write_lock:
                for rec in records:
                    out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                counts["sections"] += len(records)
                counts["processed"] += 1
                if not records:
                    counts["errors"] += 1
            with checkpoint_lock:
                with open(CHECKPOINT, "a") as ck:
                    ck.write(fname + "\n")
            if counts["processed"] % 100 == 0 or counts["processed"] <= 5:
                print(f"[{counts['processed']}/{len(files_to_run)}] "
                      f"sections={counts['sections']} errors={counts['errors']}")

print(f"Done. Processed={counts['processed']} Sections={counts['sections']} Errors={counts['errors']}")

Resuming: 749 PDFs already processed
Remaining: 165 PDFs to process
[1/165] sections=20 errors=0
[2/165] sections=44 errors=0
[3/165] sections=72 errors=0
[4/165] sections=102 errors=0
[5/165] sections=122 errors=0


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1156: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /opt/pytorch/pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1479.)
  result = _VF.lstm(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1156: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /opt/pytorch/pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1479.)
  result = _VF.lstm(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1156: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be comp

## 7. Inspect output

 Reads and prints the first 5 records from the output JSONL as a quick sanity check.


In [ ]:
with open(OUT_JSONL, encoding="utf-8") as f:
    sample = [json.loads(l) for l in f.readlines()[:5]]

for rec in sample:
    print(f"bulletin_id   : {rec['bulletin_id']}")
    print(f"crop          : {rec['crop']}")
    print(f"date          : {rec['date']}")
    print(f"section_title : {rec['section_title']}")
    print(f"text preview  : {rec['text'][:120]!r}")
    print(f"images        : {len(rec['images'])}")
    print(f"tables        : {len(rec['tables'])}")
    print()


Counts total lines in the JSONL to report total sections extracted:

In [ ]:
import re
with open("bsv_parser.py") as f:
    src = f.read()
fns = re.findall(r'^def (\w+)', src, re.MULTILINE)
print(fns)

In [ ]:
with open(OUT_JSONL) as f:
    total = sum(1 for _ in f)
print(f"Total sections: {total}")

 Draws a reproducible random 500-section sample  from the full JSONL and saves it to `bsv_sample_500.jsonl` for manual curation.section.

In [ ]:
import json

with open(str(BSV_ROOT / 'Ground Truth' / 'bsv_dataset_gt_tables_docling.jsonl'), encoding="utf-8") as f:
    seen = set()
    total_found_img = total_kept_img = 0
    total_found_tbl = total_kept_tbl = 0
    for line in f:
        rec = json.loads(line)
        if rec["bulletin_id"] in seen:
            continue  # only count once per bulletin, meta is repeated per section
        seen.add(rec["bulletin_id"])
        total_found_img += rec.get("images_found_total", 0)
        total_kept_img  += rec.get("images_kept", 0)
        total_found_tbl += rec.get("tables_found_total", 0)
        total_kept_tbl  += rec.get("tables_kept", 0)

print(f"Image recall: {total_kept_img}/{total_found_img} "
      f"({100*total_kept_img/max(total_found_img,1):.1f}%)")
print(f"Table recall: {total_kept_tbl}/{total_found_tbl} "
      f"({100*total_kept_tbl/max(total_found_tbl,1):.1f}%)")

In [ ]:
import json, random
sys.path.insert(0, str(BSV_ROOT / "Scripts"))
from bsv_pair_constructor import generate_pairs

all_pairs = []
with open(str(BSV_ROOT / 'Ground Truth' / 'bsv_dataset_gt_tables_realspans.jsonl'), encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        pairs = generate_pairs(rec)
        all_pairs.extend(p for p in pairs if p["pair_type"] == "text_table")

print(f"Total text_table pairs across all 2,404 PDFs: {len(all_pairs)}")

random.seed(42)
sample_size = min(900, len(all_pairs))
sample_900 = random.sample(all_pairs, sample_size)

with open(str(BSV_ROOT / "Ground Truth" / "bsv_pairs_gt_texttable_900_realspans.jsonl"), "w", encoding="utf-8") as f:
    for p in sample_900:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"Sampled {len(sample_900)} pairs -> bsv_pairs_gt_texttable_900_realspans.jsonl")


In [ ]:
wrapper_path = BSV_ROOT / "Scripts" / "generate_reviewer_gt_texttable.py"
wrapper_path.write_text(
    "import os, runpy\n"
    "from pathlib import Path\n"
    "ROOT = Path(__file__).resolve().parents[2]\n"
    "DATASET_ROOT = ROOT / '3.Dataset Construction'\n"
    "os.environ['PHYTOPAIRS_REVIEW_PAIRS'] = str(DATASET_ROOT / 'Ground Truth' / 'bsv_pairs_gt_texttable_900_realspans.jsonl')\n"
    "os.environ['PHYTOPAIRS_REVIEW_IMAGES'] = str(DATASET_ROOT / 'Ground Truth' / 'bsv_images_gt_tables_docling')\n"
    "os.environ['PHYTOPAIRS_REVIEW_HTML'] = str(DATASET_ROOT / 'Ground Truth' / 'bsv_reviewer_gt_texttable_900_realspans.html')\n"
    "runpy.run_path(str(DATASET_ROOT / 'Scripts' / 'generate_reviewer_V2.py'), run_name='__main__')\n",
    encoding="utf-8",
)
print(f"Wrote reviewer launcher: {wrapper_path}")
